In [ ]:
"""
Generating with a 7B model on a Colab T4 can be painfully slow, especially when processing sequences that are thousands of tokens long. While bitsandbytes 4-bit quantization (NF4) is heavily optimized for memory savings during QLoRA training—which makes it excellent for fine-tuning pipelines—it is notoriously slow for pure inference because the weights have to be continuously dequantized on the fly.

Here is how you can dramatically speed up your inference on a T4:

Switch to AWQ or GPTQ Quantization: Instead of loading the base model and quantizing it dynamically with bitsandbytes, download a pre-quantized model. Look for Qwen/Qwen2.5-7B-Instruct-AWQ on Hugging Face. AWQ (Activation-aware Weight Quantization) is designed specifically for fast inference and will yield a noticeable speed boost over NF4 while maintaining similar quality.

Use vLLM: Standard Hugging Face generate() is not optimized for speed. If you wrap your generation pipeline in a high-throughput serving engine like vLLM (which uses PagedAttention to manage key-value cache memory efficiently), you can often see a 3x to 5x speedup, even on a single T4.

Enable SDPA (Scaled Dot Product Attention): In your model_kwargs, you can add "attn_implementation": "sdpa". The T4 GPU doesn't fully support the hardware-level Flash Attention 2, but PyTorch's native SDPA provides a memory-efficient attention mechanism that will still speed up processing for long context windows.

Implement Batching: You are currently iterating through the dataset one sample at a time. The T4 GPU shines when it can process matrices in parallel. Pass a list of 2 to 4 prompts to the tokenizer and model simultaneously. Keep in mind that for batching to work properly, you will need to set tokenizer.padding_side = "left" and ensure you are passing the attention_mask to the generate() function.
"""

In [9]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


In [10]:
def check_system_info():
    """Display system information for debugging"""
    print("=" * 60)
    print("SYSTEM INFORMATION")
    print("=" * 60)
    print(f"PyTorch Version: {torch.__version__}")
    print(f"CUDA Available: {torch.cuda.is_available()}")

    if torch.cuda.is_available():
        print(f"CUDA Version: {torch.version.cuda}")
        print(f"GPU Device: {torch.cuda.get_device_name(0)}")
        print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    else:
        print("Running on CPU (slower performance)")
    print("=" * 60)

check_system_info()

SYSTEM INFORMATION
PyTorch Version: 2.10.0+cu128
CUDA Available: True
CUDA Version: 12.8
GPU Device: Tesla T4
GPU Memory: 14.56 GB


In [11]:
# Configuration - Change this based on your hardware
USE_QUANTIZATION = '4bit'  # Options: None, '8bit', '4bit'

print(f"Configuration: Quantization = {USE_QUANTIZATION if USE_QUANTIZATION else 'None (Full Precision)'}")

Configuration: Quantization = 4bit


In [12]:
from transformers import BitsAndBytesConfig

# Configure model loading based on quantization
model_kwargs = {
    "trust_remote_code": True,
    "device_map": "auto"
}

In [13]:
def load_model(use_quantization=None):
    """
    Load Qwen 2.5 7B Instruct model

    Args:
        use_quantization: None, '8bit', or '4bit'
    """
    model_name = "Qwen/Qwen2.5-7B-Instruct"

    print(f"Loading model: {model_name}")
    print(f"Quantization: {use_quantization if use_quantization else 'None (FP16/BF16)'}")
    print("This may take a few minutes on first run...")
    print()

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True
    )

    # Configure model loading based on quantization
    model_kwargs = {
        "trust_remote_code": True,
        "device_map": "auto"  # Automatically distribute across available devices
    }

    if use_quantization == '8bit':
        print("Loading with 8-bit quantization (requires bitsandbytes)")
        model_kwargs["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
    elif use_quantization == '4bit':
        print("Loading with 4-bit quantization (requires bitsandbytes)")
        model_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,   # faster compute
            bnb_4bit_use_double_quant=True,            # saves a bit more VRAM
            bnb_4bit_quant_type="nf4"                  # NF4 is standard for QLoRA/inference
        )
    else:
        if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
            model_kwargs["torch_dtype"] = torch.bfloat16
            print("Using BF16 precision")
        else:
            model_kwargs["torch_dtype"] = torch.float16
            print("Using FP16 precision")

    # Load model
    # pip install -U bitsandbytes>=0.46.1, for colab environment
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        **model_kwargs
    )

    print("Model loaded successfully!")

    return model, tokenizer

# Load the model
model, tokenizer = load_model(use_quantization=USE_QUANTIZATION)

Loading model: Qwen/Qwen2.5-7B-Instruct
Quantization: 4bit
This may take a few minutes on first run...

Loading with 4-bit quantization (requires bitsandbytes)


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Model loaded successfully!


In [15]:
def generate_response(model, tokenizer, prompt, max_new_tokens=512, temperature=0.7):
    """Generate a response from the model"""

    # Format the prompt using chat template
    messages = [
        {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize
    model_inputs = tokenizer([text], return_tensors="pt")

    # Move to same device as model
    if torch.cuda.is_available():
        model_inputs = model_inputs.to(model.device)

    # Generate
    print("Generating response...")
    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # Decode only the generated tokens (exclude the prompt)
    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    return response

print("Generation function defined successfully!")

Generation function defined successfully!


In [ ]:
"""
# Example 1: Fibonacci sequence
prompt1 = "Introduce the linear regression."
# prompt1 = 'Write a python function to print hello world'
print("=" * 60)
print(f"Prompt: {prompt1}")
print("=" * 60)

response1 = generate_response(
    model,
    tokenizer,
    prompt1,
    max_new_tokens=512,
    temperature=0.7
)

print("\nResponse:")
print(response1)
print("=" * 60)
"""

In [25]:
# Install datasets library if not present
# !pip install -q datasets rouge-score bert-score

from datasets import load_dataset

dataset = load_dataset("ccdv/govreport-summarization")
print(dataset)
print("\nSample keys:", dataset["train"][0].keys())
print("Doc length (chars):", len(dataset["train"][0]["report"]))
print("Summary length (chars):", len(dataset["train"][0]["summary"]))


DatasetDict({
    train: Dataset({
        features: ['report', 'summary'],
        num_rows: 17517
    })
    validation: Dataset({
        features: ['report', 'summary'],
        num_rows: 973
    })
    test: Dataset({
        features: ['report', 'summary'],
        num_rows: 973
    })
})

Sample keys: dict_keys(['report', 'summary'])
Doc length (chars): 62042
Summary length (chars): 4074


In [26]:
def truncate_report(report, tokenizer, max_input_tokens=3500):
    """
    Truncate report to fit within model's practical context window.
    3500 leaves room for the prompt template + generated summary.
    """
    tokens = tokenizer.encode(report, add_special_tokens=False)
    if len(tokens) > max_input_tokens:
        tokens = tokens[:max_input_tokens]
        report = tokenizer.decode(tokens, skip_special_tokens=True)
        print(f"  [Truncated to {max_input_tokens} tokens]")
    return report

def build_summarization_prompt(report):
    """Build a clear summarization prompt."""
    return (
        "Please provide a concise and accurate summary of the following government report. "
        "Focus on the key findings, recommendations, and policy implications.\n\n"
        f"REPORT:\n{report}\n\n"
        "SUMMARY:"
    )


In [27]:
# Reload model here if you cleared it — or keep model loaded from cell-4
# model, tokenizer = load_model(use_quantization=USE_QUANTIZATION)

# Use test split; start with a small batch to verify
test_data = dataset["test"]
NUM_SAMPLES = 5  # Increase after verifying correctness

results = []

for i in range(NUM_SAMPLES):
    sample = test_data[i]
    print(f"\n{'='*60}")
    print(f"Sample {i+1}/{NUM_SAMPLES}")

    # Truncate the source report
    truncated_doc = truncate_report(sample["report"], tokenizer, max_input_tokens=3500)

    # Build prompt and generate
    prompt = build_summarization_prompt(truncated_doc)
    generated_summary = generate_response(
        model, tokenizer,
        prompt,
        max_new_tokens=400,   # GovReport reference summaries avg ~600 words
        temperature=0.3        # Lower temp for factual summarization
    )

    results.append({
        "id": i,
        "report": sample["report"],       # original (untruncated) for reference
        "reference_summary": sample["summary"],
        "generated_summary": generated_summary,
    })

    print(f"Reference (first 200 chars): {sample['summary'][:200]}...")
    print(f"Generated (first 200 chars): {generated_summary[:200]}...")

print(f"\nGenerated {len(results)} summaries.")



Sample 1/5
  [Truncated to 3500 tokens]
Generating response...
Reference (first 200 chars): Multiple firms have produced cell-cultured meat as part of their research and development. These products appear likely to become available to consumers in coming years. FDA and USDA are the primary a...
Generated (first 200 chars): The report highlights the journey of technological innovation from invention to commercialization, emphasizing the inherent risks involved. It focuses on the regulatory landscape of the U.S. Departmen...

Sample 2/5
  [Truncated to 3500 tokens]
Generating response...
Reference (first 200 chars): Federal advisory committees provide advice to federal agencies on many topics. As of March 31, 2018, EPA managed 22 such committees. They advise the agency on such issues as developing regulations and...
Generated (first 200 chars): The government report focuses on the establishment and operation of advisory committees managed by the Environmental Protection Agency (EPA) und

In [29]:
!pip install rouge-score bert-score

from bert_score import score as bert_score
from rouge_score import rouge_scorer

def compute_rouge(results):
    scorer = rouge_scorer.RougeScorer(
        ["rouge1", "rouge2", "rougeL"], use_stemmer=True
    )

    all_scores = {"rouge1": [], "rouge2": [], "rougeL": []}

    for r in results:
        scores = scorer.score(r["reference_summary"], r["generated_summary"])
        for key in all_scores:
            all_scores[key].append(scores[key].fmeasure)

    print("=" * 50)
    print("ROUGE SCORES (F1, averaged over samples)")
    print("=" * 50)
    for key, values in all_scores.items():
        avg = sum(values) / len(values)
        print(f"  {key.upper()}: {avg:.4f}")

    return all_scores

rouge_scores = compute_rouge(results)


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.6 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=30a5e3667ee13255893fcb086d7e5d002231d1ce66b036294e193e67333774e4
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score
ROUGE SCORES (F1, averaged over samples)
  ROUGE1: 0.4051
  ROUGE2: 0.1119
  ROUGEL: 0.1711


In [30]:
from bert_score import score as bert_score

def compute_bertscore(results):
    references = [r["reference_summary"] for r in results]
    candidates = [r["generated_summary"] for r in results]

    # Uses DeBERTa by default; rescale_with_baseline improves interpretability
    P, R, F1 = bert_score(
        candidates, references,
        lang="en",
        rescale_with_baseline=True,
        verbose=True
    )

    print("=" * 50)
    print("BERTScore (averaged over samples)")
    print("=" * 50)
    print(f"  Precision: {P.mean().item():.4f}")
    print(f"  Recall:    {R.mean().item():.4f}")
    print(f"  F1:        {F1.mean().item():.4f}")

    return {"precision": P, "recall": R, "f1": F1}

bert_scores = compute_bertscore(results)


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 1.19 seconds, 4.21 sentences/sec
BERTScore (averaged over samples)
  Precision: 0.0600
  Recall:    0.0924
  F1:        0.0775


In [32]:
def self_check_summary(model, tokenizer, report, summary):
    """
    Ask the model to verify if the summary is faithful to the source report.
    Returns a score string: 'Faithful', 'Partially Faithful', or 'Unfaithful'.
    """
    # Use a truncated doc for the faithfulness check
    truncated_doc = truncate_report(report, tokenizer, max_input_tokens=2500)

    verification_prompt = (
        "Given the following source report and its summary, "
        "evaluate whether the summary is factually faithful to the report.\n\n"
        f"report:\n{truncated_doc}\n\n"
        f"SUMMARY:\n{summary}\n\n"
        "Instructions: Does the summary contain only information that is supported by "
        "the report? Answer with one of: 'Faithful', 'Partially Faithful', or 'Unfaithful'. "
        "Then briefly explain why in 1-2 sentences.\n\n"
        "Verdict:"
    )

    verdict = generate_response(
        model, tokenizer,
        verification_prompt,
        max_new_tokens=100,
        temperature=0.1   # Near-deterministic for binary judgment
    )
    return verdict.strip()


# Run self-check on the generated summaries
print("Running self-checking faithfulness evaluation...")
for i, r in enumerate(results):
    verdict = self_check_summary(
        model, tokenizer,
        r["report"],
        r["generated_summary"]
    )
    r["faithfulness_verdict"] = verdict
    print(f"\nSample {i+1}: {verdict}")


Running self-checking faithfulness evaluation...
  [Truncated to 2500 tokens]
Generating response...

Sample 1: Faithful

The summary accurately captures the key points from the report, including the regulatory roles of FDA and USDA, the process of cell-cultured meat production, and the challenges in adapting existing regulatory frameworks. It also appropriately highlights the uncertainties and policy implications related to cell-cultured meat.
  [Truncated to 2500 tokens]
Generating response...

Sample 2: Faithful

The summary accurately captures the key points from the report, including the importance of balanced membership, ethics requirements, types of advisory committees, the appointment process, financial disclosure, and the aim of enhancing transparency and accountability. The summary does not introduce any unsupported information or distortions from the original report.
  [Truncated to 2500 tokens]
Generating response...

Sample 3: Faithful

The summary accurately captures the 

In [33]:
import pandas as pd
from rouge_score import rouge_scorer

def full_evaluation_report(results):
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    rows = []

    for r in results:
        s = scorer.score(r["reference_summary"], r["generated_summary"])
        rows.append({
            "sample_id": r["id"],
            "rouge1_f1": round(s["rouge1"].fmeasure, 4),
            "rouge2_f1": round(s["rouge2"].fmeasure, 4),
            "rougeL_f1": round(s["rougeL"].fmeasure, 4),
            "faithfulness": r.get("faithfulness_verdict", "N/A")[:50],
            "gen_length": len(r["generated_summary"].split()),
            "ref_length": len(r["reference_summary"].split()),
        })

    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    print("\nAverages:")
    print(df[["rouge1_f1", "rouge2_f1", "rougeL_f1", "gen_length", "ref_length"]].mean())
    return df

df_report = full_evaluation_report(results)


 sample_id  rouge1_f1  rouge2_f1  rougeL_f1                                         faithfulness  gen_length  ref_length
         0     0.4357     0.1227     0.2130 Faithful\n\nThe summary accurately captures the key          290         500
         1     0.3731     0.1045     0.1464 Faithful\n\nThe summary accurately captures the key          295         680
         2     0.4221     0.0950     0.1558 Faithful\n\nThe summary accurately captures the main         312         553
         3     0.4309     0.1323     0.1843 Faithful\n\nThe summary accurately captures the key          273         523
         4     0.3636     0.1050     0.1561 Faithful\n\nThe summary accurately captures the main         302         617

Averages:
rouge1_f1       0.40508
rouge2_f1       0.11190
rougeL_f1       0.17112
gen_length    294.40000
ref_length    574.60000
dtype: float64


In [8]:
"""
# Clear GPU memory
import gc

# Delete model and tokenizer
del model
del tokenizer

# Run garbage collection
gc.collect()

# Clear CUDA cache if using GPU
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU memory cleared")

print("Memory freed successfully!")
"""

GPU memory cleared
Memory freed successfully!
